# Renewables Ninja ; Solar PV Generation Analysis

This notebook fetches real hourly solar PV generation data for any 
location in the world using the Renewables Ninja API ; powered by 
NASA MERRA2 reanalysis data.

**What this notebook does:**
- Takes a location name and converts it to coordinates using geopy
- Authenticates with the Renewables Ninja API using a token
- Fetches hourly solar PV generation data for a full year
- Loads the data into a pandas DataFrame for analysis
- Plots hourly solar electricity output

**The goal simplified:**
Asking Renewables Ninja ; "how much electricity would a 1MW solar PV 
system generate, hour by hour, for an entire year, at a location 
I choose?"

**Why this is powerful:**
In the real world, before anyone builds a solar farm, they need to answer:

- How much energy will this system actually produce?
- What time of year produces the most solar energy?
- How does irradiance and temperature affect output?
- Is this location suitable for solar investment?

Renewables Ninja answers all of these questions using real NASA 
satellite weather data ; without needing to install any physical 
equipment first.

---

## Step 1 : Setup

Loading all required libraries and reading the API token 
securely from the .env file.

---

In [4]:
# Standard libraries
import requests       
import json           

# Data and visualisation
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

# Location
from geopy.geocoders import Nominatim

# Security ; reads token from .env file
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv()

# Read the token
token = os.getenv("NINJA_TOKEN")

# Confirm token loaded ; never print the actual token!
print("Token loaded successfully:", token is not None)

Token loaded successfully: True


## Step 2 : Location

To fetch data from the Renewables Ninja API, I need coordinates not a city name. I used geopy here is the documentation -> (https://geopy.readthedocs.io) to automatically convert any location I type into latitude and longitude.

I chose Nicosia, Cyprus as my starting location.

In [6]:
# Initialise the geocoder
geolocator = Nominatim(user_agent="renewables_ninja_app")

# Ask user for a location
# place = input("Enter a location as  city, country : ")

# Convert city name to coordinates
# information = geolocator.geocode(place)

information = geolocator.geocode("Nicosia, Cyprus")

# Extract latitude and longitude
lat = information[1][0]
lon = information[1][1]

# print(f"Location : {place}")
print(f"Latitude : {lat}")
print(f"Longitude : {lon}")

Latitude : 35.1746503
Longitude : 33.3638783


---
Nicosia, Cyprus returned coordinates:
- Latitude : 35.1746503
- Longitude : 33.3638783

These coordinates will be passed directly to the Renewables Ninja 
API to fetch solar and wind generation data specific to this location.

---

## Step 3 : Fetch Solar Data

I configured a 1MW solar PV system at the Nicosia location 
and called the Renewables Ninja API to fetch hourly generation 
data for the full year 2020.

**System configuration:**
- Capacity : 1000 kW (1 MW)
- Tilt : 35° ; optimal for latitude 35°N (this is a well known rule of thumb in solar engineering: Optimal tilt angle ≈ latitude of the location)
- Azimuth : 180° ; south facing for Northern Hemisphere
- System loss : 20%
- Tracking : fixed ; no tracking
- Dataset : NASA MERRA2

---

In [7]:
# API base URL
api_base = "https://www.renewables.ninja/api/"

# Solar PV endpoint
url_solar = api_base + "data/pv"

# Solar PV parameters
args_solar = {
    "lat": lat,
    "lon": lon,
    "date_from": "2020-01-01",
    "date_to": "2020-12-31",
    "dataset": "merra2",
    "capacity": 1000,
    "system_loss": 0.2,
    "tracking": 0,
    "tilt": 35,
    "azim": 180,
    "format": "json",
    "local_time": "true",
    "raw": "true"
}

# Create a session and authenticate with token
session = requests.session()
session.headers = {"Authorization": "Token " + token}

# Make the API call
print("Fetching solar data...")
response_solar = session.get(url_solar, params=args_solar)

# Check if request was successful
print(f"Status code : {response_solar.status_code}")
print("200 = success ; 404 = not found ; 403 = unauthorised")

Fetching solar data...
Status code : 200
200 = success ; 404 = not found ; 403 = unauthorised


In [11]:
import io

# Parse the JSON response into Python
parsed_solar = json.loads(response_solar.text)

# Extract metadata
metadata_solar = parsed_solar["metadata"]

# Convert data to DataFrame using StringIO
data_solar = pd.read_json(
    io.StringIO(json.dumps(parsed_solar["data"])), 
    orient="index"
)

# Confirm what we got
print("Metadata:")
print(metadata_solar)
print(f"\nData shape : {data_solar.shape}")
print(f"Columns : {list(data_solar.columns)}")
data_solar.head()


Metadata:
{'units': {'time': 'UTC', 'local_time': 'Europe/Athens', 'electricity': 'kW', 'irradiance_direct': 'unknown', 'irradiance_diffuse': 'unknown', 'temperature': 'ºC'}, 'params': {'lat': '35.1746503', 'lon': '33.3638783', 'date_from': '2020-01-01', 'date_to': '2020-12-31', 'dataset': 'merra2', 'capacity': '1000', 'system_loss': '0.2', 'tracking': '0', 'tilt': '35', 'azim': '180', 'local_time': 'true', 'raw': 'true'}}

Data shape : (8784, 5)
Columns : ['local_time', 'electricity', 'irradiance_direct', 'irradiance_diffuse', 'temperature']


,local_time,electricity,irradiance_direct,irradiance_diffuse,temperature
2020-01-01 00:00:00,2020-01-01 02:00:00+02:00,0.0,0.0,0.0,12.782
2020-01-01 01:00:00,2020-01-01 03:00:00+02:00,0.0,0.0,0.0,12.776
2020-01-01 02:00:00,2020-01-01 04:00:00+02:00,0.0,0.0,0.0,12.673
2020-01-01 03:00:00,2020-01-01 05:00:00+02:00,0.0,0.0,0.0,12.545
2020-01-01 04:00:00,2020-01-01 06:00:00+02:00,0.0,0.0,0.0,12.349


The API returned 8,784 rows rather than 8,760 because 2020 was a leap year ; 366 days × 24 hours = 8,784 hourly records.

The first row confirms the data is in local Cyprus time (UTC+2).Electricity output at midnight is 0.0 kW as expected ; no solar generation without sunlight.

Temperature at midnight in January reads 12.78°C ; consistent with North Cyprus winter conditions.